# PROJECT WORKFLOW

## Preprocessing :
## Transformer en data frame 
## Corrélation, regroupe colonnes ensemble
## Normaliser
## Mettre des poids pcq la data est pas symétrique (ex 10000 negatif, 100 positif donc déséquilibré)
## f1_score
## disitributions
## outliers ?
## NaN, remplacer les valeurs ?
## traduire/intérpêter les données 

## Feature engineering :
## 

SETUP AND INPORTS

In [32]:
import numpy as np
import matplotlib.pyplot as plt
from src.helpers import *            # load_csv_data, create_csv_submission
from src.implementations import *    # your 6 functions from Table 1

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LOAD THE DATA

In [33]:
DATA_PATH = "data/"   # folder containing x_train.csv, y_train.csv, x_test.csv
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print(x_train.shape, x_test.shape, y_train.shape)
print("Fraction of positive labels:", np.mean(y_train == 1))

(328135, 321) (109379, 321) (328135,)
Fraction of positive labels: 0.08830207079403295


In [34]:
import pandas as pd

df = pd.read_csv("data/x_train.csv")

df.head(15)

,Id,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,...,_PAREC1,_PASTAE1,_LMTACT1,_LMTWRK1,_LMTSCL1,_RFSEAT2,_RFSEAT3,_FLSHOT6,_PNEUMO2,_AIDTST3
0,0,53,11,11162015,11,16,2015,1100,2015015629,2015015629,...,1,1,3.0,3.0,4.0,1,1,NaN,NaN,2.0
1,1,33,12,12152015,12,15,2015,1200,2015004387,2015004387,...,9,9,3.0,3.0,4.0,9,9,NaN,NaN,NaN
2,2,20,10,10202015,10,20,2015,1100,2015005638,2015005638,...,4,2,2.0,2.0,3.0,1,1,1.0,2.0,2.0
3,3,42,6,6182015,6,18,2015,1100,2015004694,2015004694,...,2,2,2.0,2.0,3.0,1,1,2.0,2.0,2.0
4,4,24,11,11062015,11,6,2015,1100,2015004673,2015004673,...,9,9,3.0,3.0,4.0,1,1,9.0,9.0,2.0
5,5,54,4,4082015,4,8,2015,1100,2015004111,2015004111,...,4,2,3.0,3.0,4.0,1,1,NaN,NaN,2.0
6,6,39,12,12072015,12,7,2015,1100,2015005521,2015005521,...,4,2,1.0,2.0,2.0,1,1,1.0,1.0,2.0
7,7,44,3,3072015,3,7,2015,1100,2015000377,2015000377,...,3,2,2.0,2.0,2.0,1,2,1.0,2.0,2.0
8,8,21,7,7312015,7,31,2015,1100,2015007052,2015007052,...,1,1,3.0,3.0,4.0,1,1,NaN,NaN,2.0
9,9,4,6,6232015,6,23,2015,1200,2015006238,2015006238,...,4,2,2.0,2.0,3.0,1,1,NaN,NaN,NaN


In [35]:
import pandas as pd

df = pd.DataFrame(x_train)
df["target"] = y_train

display(df.head(10))

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df["target"].value_counts())
print("\nTarget distribution (%):")
print(df["target"].value_counts(normalize=True) * 100)

,0,1,2,3,4,5,6,7,8,9,...,312,313,314,315,316,317,318,319,320,target
0,53.0,11.0,11162015.0,11.0,16.0,2015.0,1100.0,2.015016e+09,2.015016e+09,NaN,...,1.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0,-1
1,33.0,12.0,12152015.0,12.0,15.0,2015.0,1200.0,2.015004e+09,2.015004e+09,1.0,...,9.0,3.0,3.0,4.0,9.0,9.0,NaN,NaN,NaN,-1
2,20.0,10.0,10202015.0,10.0,20.0,2015.0,1100.0,2.015006e+09,2.015006e+09,1.0,...,2.0,2.0,2.0,3.0,1.0,1.0,1.0,2.0,2.0,-1
3,42.0,6.0,6182015.0,6.0,18.0,2015.0,1100.0,2.015005e+09,2.015005e+09,NaN,...,2.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0,-1
4,24.0,11.0,11062015.0,11.0,6.0,2015.0,1100.0,2.015005e+09,2.015005e+09,1.0,...,9.0,3.0,3.0,4.0,1.0,1.0,9.0,9.0,2.0,-1
5,54.0,4.0,4082015.0,4.0,8.0,2015.0,1100.0,2.015004e+09,2.015004e+09,NaN,...,2.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0,-1
6,39.0,12.0,12072015.0,12.0,7.0,2015.0,1100.0,2.015006e+09,2.015006e+09,1.0,...,2.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,1
7,44.0,3.0,3072015.0,3.0,7.0,2015.0,1100.0,2.015000e+09,2.015000e+09,1.0,...,2.0,2.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,1
8,21.0,7.0,7312015.0,7.0,31.0,2015.0,1100.0,2.015007e+09,2.015007e+09,NaN,...,1.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0,-1
9,4.0,6.0,6232015.0,6.0,23.0,2015.0,1200.0,2.015006e+09,2.015006e+09,NaN,...,2.0,2.0,2.0,3.0,1.0,1.0,NaN,NaN,NaN,-1


Shape: (328135, 322)

Target distribution:
target
-1    299160
 1     28975
Name: count, dtype: int64

Target distribution (%):
target
-1    91.169793
 1     8.830207
Name: proportion, dtype: float64


MINIMAL PREPOCESSING

In [36]:
# Replace missing values (NaN) by the column median computed on the TRAIN set
median = np.nan_to_num(np.nanmedian(x_train, axis=0))
x_train = np.where(np.isnan(x_train), median, x_train)
x_test = np.where(np.isnan(x_test), median, x_test)

# Standardize with train statistics
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
std[std == 0] = 1                     # avoid division by zero
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

# Add the offset (bias) column -> tx, as in the labs
tx_train = np.c_[np.ones(x_train.shape[0]), x_train]
tx_test = np.c_[np.ones(x_test.shape[0]), x_test]

# Labels in {0,1} for logistic regression
y = (y_train + 1) / 2

LOCAL TRAIN/VALIDATION SPLIT

In [37]:
def split_data(y, tx, ratio, seed=1):
    """Randomly split the data into train and validation sets."""
    np.random.seed(seed)
    idx = np.random.permutation(len(y))
    cut = int(ratio * len(y))
    return tx[idx[:cut]], y[idx[:cut]], tx[idx[cut:]], y[idx[cut:]]

tx_tr, y_tr, tx_val, y_val = split_data(y, tx_train, ratio=0.8)

TRAIN A MODEL

In [38]:
initial_w = np.zeros(tx_tr.shape[1])
max_iters = 200
gamma = 0.1
lambda_ = 1e-4

w, loss = reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)
print("train loss:", loss)

train loss: 0.24262476280379847


EVALUATE ON VALIDATION SET

In [39]:
def sigmoid(t):
    return 1.0 / (1.0 + np.exp(-t))

def predict_labels(w, tx, threshold=0.5):
    """Return predictions in {-1, 1} (format required by the submission)."""
    return np.where(sigmoid(tx @ w) >= threshold, 1, -1)

def f1_score(y_true, y_pred):
    """F1 score for labels in {-1, 1}."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return 2 * tp / (2 * tp + fp + fn + 1e-12)

y_val_pm = 2 * y_val - 1             # back to {-1, 1}
y_pred_val = predict_labels(w, tx_val)
print("accuracy:", np.mean(y_pred_val == y_val_pm))
print("F1 score:", f1_score(y_val_pm, y_pred_val))

accuracy: 0.9157511390129063
F1 score: 0.13161614575153133


CREATE THE SUBMISSION

In [40]:
y_pred = predict_labels(w, tx_test)
create_csv_submission(test_ids, y_pred, "submission1.csv")